# Fundamentals 06 - Integrations Strands API

**Historia:** Agentic Systems define el contrato; `framework="strands"` declara el loop como integracion agnostica.

La fachada sigue empezando en Agentic Systems:

```text
lab.agent(..., framework="strands") → await agent.arun(...)
```


> 2.4.6: las integraciones viven en `agentic_systems.integrations.*`; `agentic_systems.bridges.*` queda sólo como ruta explicable.


In [ ]:
import agentic_systems as lab

PRETTY = False  # Cambia a True para usar Rich; False imprime texto plano estable y reproducible.

scheduler = lab.scheduler(timeout_s=60, max_retries=0, max_tool_calls=4, max_turns=8)
runtime = lab.runtime(provider="auto", scheduler=scheduler)
runtime_description = runtime.describe()

lab.show({
    "runtime_auto_resolution": runtime_description,
    "human_results_pretty": PRETTY,
})

## Problema default de fundamentals

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: sólo representa la sección `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Problema default · visible")


## 1) Definir tools explícitas


In [ ]:
@lab.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos números."""
    return {"operation": "sumar", "result": a + b}

@lab.tool
def restar(a: int, b: int) -> dict:
    """Resta dos números."""
    return {"operation": "restar", "result": a - b}

@lab.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos números."""
    return {"operation": "multiplicar", "result": a * b}

@lab.tool
def dividir(a: int, b: int) -> dict:
    """Divide dos números."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    return {"operation": "dividir", "result": int(value) if value.is_integer() else value}

tools = [sumar, restar, multiplicar, dividir]

lab.show({
    "tools": [tool.name for tool in tools],
    "mental_model": "tools explícitas → agent = lab.agent(..., tools=tools)",
})


## 2) Crear agente portable con framework Strands

Aqu? se ven separadas las dos decisiones:

```text
runtime   = backend de inferencia/modelo seleccionado por provider="auto"
framework = loop/adaptador de agente
```

Primero se declara `runtime`. Despu?s el framework obedece ese runtime: `framework="strands"` no decide el backend, solo declara el adaptador.

In [ ]:
instructions = """
Eres un agente calculadora.
Usa las tools disponibles para resolver la solicitud con evidencia auditable.
"""

workspace = lab.AgenticSystem(
    model=runtime.model_id or lab.default_model_id(),
    region=runtime.region_name or lab.default_region(),
    runtime=runtime,
)

agent = workspace.agent(
    name="calculator_strands_integration",
    instructions=instructions,
    tools=tools,
    runtime=runtime,
    framework="strands",
)

lab.show({
    "agent_name": agent.name,
    "engine": agent.engine,
    "framework": agent.framework,
    "auto_resolution": runtime_description,
    "tools": [tool.name for tool in tools],
    "what_is_hidden": "nada de negocio: solo el adapter del loop externo",
    "what_is_still_agentic_systems": ["runtime", "tools", "contract/result envelope", "human_result", "lineage"],
})

## 3) Ejecutar con la fachada Agentic Systems

La ejecuci?n se mantiene agn?stica: si `provider="auto"` no encuentra backend configurado, el notebook salta la llamada externa. El contrato del agente y el framework siguen siendo visibles.

In [ ]:
user_prompt = USER_PROMPT

if runtime_description["selected_provider"] == "auto":
    result = None
    lab.show({
        "status": "skipped",
        "reason": runtime_description["reason"],
        "how_to_enable": "Configura OPENAI_API_KEY o credenciales/configuraci?n Bedrock antes de abrir el kernel.",
    }, title="Strands framework integration saltada")
else:
    result = agent.run(user_prompt, mode="eval")

    lab.human_result(
        result,
        title="Human result - Strands framework facade integration",
        expected_tools=lab.expect.all_of("sumar", "restar", "multiplicar", "dividir"),
        pretty=PRETTY,
    )

## 4) Definir `finalize`: evidencia → salidas solicitadas

`finalize` no es otra tool y no llama LLM. Es el cierre del loop: toma la evidencia estructurada de las tools y rellena lo que el usuario pidió (`procedimiento`, `resultado_final`).

La respuesta libre del agente puede ser útil como narración, pero la salida final auditable se arma desde los tool outputs.

In [ ]:
def _tool_data(tool: dict) -> dict:
    output = tool.get("output") if isinstance(tool, dict) else {}
    if isinstance(output, dict) and isinstance(output.get("data"), dict):
        return output["data"]
    return output if isinstance(output, dict) else {}


def _procedure_from_tools(tools: list[dict]) -> list[str]:
    symbols = {"sumar": "+", "restar": "-", "multiplicar": "?", "dividir": "?"}
    lines = []
    for tool in tools:
        name = tool.get("name")
        payload = _tool_data(tool)
        tool_input = tool.get("input") if isinstance(tool.get("input"), dict) else {}
        result_value = payload.get("result", payload.get("value"))
        if name in symbols and {"a", "b"}.issubset(tool_input) and result_value is not None:
            lines.append(f"{tool_input['a']} {symbols[name]} {tool_input['b']} = {result_value}")
    return lines


def finalize(run_result, requested_outputs: list[str]) -> dict:
    normalized = run_result.normalized() if hasattr(run_result, "normalized") else {}
    tools = normalized.get("tools") if isinstance(normalized.get("tools"), list) else []
    procedure = _procedure_from_tools(tools)

    final_value = None
    for tool in reversed(tools):
        data = _tool_data(tool)
        final_value = data.get("result", data.get("value"))
        if final_value is not None:
            break

    requested = {}
    if "procedimiento" in requested_outputs:
        requested["procedimiento"] = procedure
    if "resultado_final" in requested_outputs:
        requested["resultado_final"] = final_value

    return {
        "requested_outputs": requested_outputs,
        "final_answer": requested,
        "source": "structured_tool_outputs",
        "agent_text_observation": (normalized.get("answer") or {}).get("text"),
    }


if result is None:
    lab.show({"status": "skipped", "reason": "no hay resultado LM para finalizar"}, title="Finalize saltado")
else:
    finalized = finalize(result, REQUESTED_OUTPUTS)
    lab.show(finalized, title="Finalize ? evidencia ? salidas solicitadas")

## 5) Lineage Memory transversal del loop externo

El resultado sigue siendo `RunResult`, pero el adapter permite proyectar una traza de Strands o un resultado envuelto por Agentic Systems al mismo formato de explicación.

In [ ]:
if result is None:
    lab.show({"status": "skipped", "reason": "no hay resultado LM para lineage"}, title="Lineage Memory saltado")
else:
    lineage = result.lineage(
        name="fundamentals.calculator.strands",
        question=user_prompt,
        goal="Explicar una fachada Strands con la misma API de Lineage Memory.",
    )

    lab.show(lineage, title="Lineage Memory ? Strands")

    lab.human_result(
        result,
        title="Human result + Lineage Memory ? Strands",
        expected_tools=lab.expect.all_of("sumar", "restar", "multiplicar", "dividir"),
        pretty=PRETTY,
        show_lineage=True,
        lineage=lineage,
    )

## Lo importante

- Agentic Systems sigue siendo la fachada p?blica.
- `runtime(provider="auto")` selecciona backend por entorno.
- `framework="strands"` cambia el loop/framework, no el provider.
- `finalize(...)` materializa las salidas solicitadas desde evidencia estructurada, no desde texto libre.
- El resultado sigue siendo explicable con `lab.human_result`.

## Coverage API de este notebook

Esta tabla deja explícito qué parte de Agentic Systems queda materializada aquí.

In [ ]:
api_coverage = [
    {
        "api": 'runtime(provider="auto")',
        "description": "Selecciona backend por configuracion del ambiente antes de declarar el framework."
    },
    {
        "api": 'framework="strands"',
        "description": "Declara Strands como framework sin decidir el provider."
    },
    {
        "api": "agent.run",
        "description": "Demuestra ejecucion del agente con runtime y framework declarados."
    },
    {
        "api": "finalize(result, requested_outputs)",
        "description": "Cierra la salida con una funcion explicita y auditable."
    },
    {
        "api": "RunResult.lineage",
        "description": "Proyecta cualquier RunResult canonico a Lineage Memory."
    },
    {
        "api": "human_result(show_lineage=True)",
        "description": "Materializa esta parte de la API con un ejemplo directo y verificable."
    },
    {
        "api": "default arithmetic prompt",
        "description": "Usa el mismo prompt base para comparar comportamiento entre notebooks."
    }
]

lab.show({'notebook': '06_integrations_strands_api.ipynb', 'api_coverage': api_coverage})